[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SeyhaLite/ml-foundations/blob/main/docs/notebooks/02_chat_templates.ipynb)

# ០២ · Chat Templates និង enable_thinking

In [ ]:
# ⚙️ Setup — រត់ cell នេះមុនគេ (Auto-setup for Google Colab)
import os, subprocess, sys

if "google.colab" in sys.modules:
    os.chdir("/content")
    if not os.path.isdir("ml-foundations"):
        subprocess.run(["git", "clone", "-q",
                        "https://github.com/SeyhaLite/ml-foundations.git"], check=True)
    os.chdir("ml-foundations")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], check=True)

# ធ្វើឱ្យ qwen_ft import បានភ្លាមៗក្នុង session នេះ (មិនបាច់ restart)
for _cand in ("src", os.path.join("..", "..", "src")):
    _cand = os.path.abspath(_cand)
    if os.path.isdir(_cand) and _cand not in sys.path:
        sys.path.insert(0, _cand)

import qwen_ft
print("qwen-ft", qwen_ft.__version__, "ready ✅")

## ហេតុអ្វីត្រូវការ Chat Template?

LLM ត្រូវបានហ្វឹកឱ្យស្គាល់ **ទម្រង់សន្ទនា** ជាក់លាក់ (special tokens ដូចជា
role markers)។ បើអ្នកបញ្ចូលអត្ថបទឆៅ — ម៉ូដែលមិនដឹងទេថាណាជាសំណួរ ណាជាចម្លើយ។

**កុំសរសេរ template ដោយដៃ** — ប្រើ `apply_chat_template()` ជានិច្ច។

## សាងសង់ messages ត្រឹមត្រូវ + ផ្ទៀងផ្ទាត់

យើងប្រើ validator របស់ `qwen_ft.data` ដើម្បីត្រួតពិនិត្យរចនាសម្ព័ន្ធ
មុនបញ្ជូនទៅម៉ូដែល (Python សុទ្ធ — គ្មាន download):

In [ ]:
from qwen_ft.data import validate_messages, DataValidationError

messages = [
    {"role": "system", "content": "អ្នកជាជំនួយការភាសាខ្មែរ។"},
    {"role": "user", "content": "តើ LoRA គឺជាអ្វី?"},
    {"role": "assistant", "content": "LoRA គឺជាបច្ចេកទេស fine-tuning..."},
    {"role": "user", "content": "ហេតុអ្វីវាសន្សំ VRAM?"},
    {"role": "assistant", "content": "ព្រោះវាបង្កកទម្ងន់ដើម..."},
]
validate_messages(messages)
print("✅ សន្ទនាត្រឹមត្រូវតាមច្បាប់ SFT")

# សាកល្បងកំហុស — សន្ទនាមិនបញ្ចប់ដោយ assistant
try:
    validate_messages([{"role": "user", "content": "សួស្តី"}])
except DataValidationError as e:
    print("❌ ចាប់បានកំហុស:", e)

## enable_thinking — តារាង default

| ម៉ូដែល | Thinking default | បើក/បិទ |
|---|---|---|
| Qwen3 (2025, 0.6B–32B) | **បើក** | `enable_thinking=False` |
| Qwen3-…-Instruct-2507 | បិទ (non-thinking) | ប្រើ `-Thinking-2507` បើចង់គិត |
| Qwen3.5 Small (0.8B–9B) | **បិទ** | `enable_thinking=True` |
| Qwen3.5 ≥27B / Qwen3.6 / 3.8 | **បើក** | `enable_thinking=False` |
| Qwen3.8 បន្ថែម | — | `reasoning_effort`, `preserve_thinking` |

```
ច្បាប់មាស: inference ត្រូវដូចការហ្វឹក!
```

## មើល template ពិតរបស់ Qwen (guarded)

```bash
export QWEN_FT_ALLOW_HEAVY=1   # បន្ទាប់ពី pip install "transformers>=5.8,<6"
```

In [ ]:
import os
ALLOW_HEAVY = os.environ.get("QWEN_FT_ALLOW_HEAVY", "0") == "1"
if ALLOW_HEAVY:
    # ប្រៀបធៀប prompt រវាង thinking on/off
    from transformers import AutoTokenizer

    tok = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-0.8B")
    for thinking in (False, True):
        prompt = tok.apply_chat_template(
            messages[:2],            # system + user
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=thinking,
        )
        print(f"--- enable_thinking={thinking} ---")
        print(prompt[:300])
        print()
else:
    print("⏭️  Heavy demo skipped — set QWEN_FT_ALLOW_HEAVY=1 (requires torch/transformers + model download).")

## សង្ខេប

1. `messages` ត្រូវត្រួតពិនិត្យមុនហ្វឹក — `validate_messages()`
2. `apply_chat_template(..., add_generation_prompt=True)` សម្រាប់សន្ទនា
3. `enable_thinking` default ខុសគ្នាតាមជំនាន់/ទំហំ — ត្រួតពិនិត្យតារាង

### លំហាត់
- សរសេរសន្ទនា ៣ ជុំ + system prompt ហើយផ្ទៀងផ្ទាត់
- បង្កើតកំហុស ៣ ប្រភេទ (role ខុស, បញ្ចប់ដោយ user, content ទទេ) — validator ចាប់បានទេ?
- អានជំពូក [៧ · វាយតម្លៃ និង Inference](../chapters/ch07-evaluation-inference.md)